# Agent-to-Agent (A2A)

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **A2A (Agent2Agent)** — the open protocol that lets independent AI agents discover each other and collaborate on tasks across vendors, frameworks, and machines.

## 1. What & Why

**A2A (Agent2Agent)** is an open protocol — introduced by Google in April 2025 and now governed by the **Linux Foundation** — for **agent-to-agent communication**. It standardizes how one agent (a *client*) discovers a second agent (a *remote agent*), hands it a task, and receives results — over plain HTTP using JSON-RPC 2.0, with streaming and webhooks for long-running work.

**The problem it solves: agents are silos.** Enterprises run agents built on different stacks (one team on LangGraph, another on CrewAI, a vendor's SaaS agent behind an API). Each speaks its own bespoke API, so wiring agent A to call agent B is custom glue every time — the same M×N integration explosion that plagued tools before MCP, but now at the *agent* layer. A2A gives every agent a common envelope for "here's a task, work on it, stream me updates, here are the results."

**The key design choice: agents stay opaque.** A2A lets two agents collaborate **without sharing their internal state, memory, tools, or prompts.** Agent B is a black box exposed only through the protocol — it advertises *what* it can do, not *how*. This preserves IP and security boundaries between organizations.

**Reach for A2A when:**
- You're composing **multiple autonomous agents** — possibly owned by different teams or vendors — into one workflow.
- A task is **long-running** (minutes to hours) and you need streaming progress or webhook callbacks, not a single blocking call.
- You want **cross-framework** interop: a LangGraph agent delegating to a CrewAI agent delegating to a hosted Vertex agent.

**Skip it when:**
- Everything lives in **one process / one framework** — just call the function or use the framework's native sub-agent/handoff mechanism. A2A's HTTP + task-lifecycle overhead buys you nothing inside a monolith.
- You need an agent to call a **tool or data source**, not another agent — that's **[[model-context-protocol]]** (MCP), a different layer. A2A and MCP are complementary, not competing (see §7).

## 2. Mental Model

Think of A2A as **hiring a contractor you'll never see the inside of.**

You (the **client agent**) find a contractor by reading their **business card** — the *Agent Card*, a JSON document at a well-known URL that lists their skills, endpoint, and how to authenticate. You send a **work order** (a *Task*) describing what you want. The contractor works on it, sends you **status updates** as they go (streaming), and finally delivers **deliverables** (*Artifacts*). You never see their workshop, their staff, or their tools — only the card, the messages, and the artifacts.

```
   Client Agent                                       Remote Agent ("opaque")
   ────────────                                       ───────────────────────
        │  1. GET /.well-known/agent-card.json              │
        │ ────────────────────────────────────────────────▶ │   discover skills,
        │ ◀──────────────────────────────────────────────── │   endpoint, auth
        │            Agent Card (JSON)                       │
        │                                                    │
        │  2. message/send  { Task: "translate this doc" }   │
        │ ────────────────────────────────────────────────▶ │   Task: submitted
        │ ◀── ── ── ── streamed status updates ── ── ── ──── │   Task: working
        │            (SSE: working… input-required…)         │   Task: input-required
        │  3. message/send  { taskId, "yes, formal tone" }   │
        │ ────────────────────────────────────────────────▶ │   Task: working
        │ ◀──────────────────────────────────────────────── │   Task: completed
        │            Artifact(s) = the result                │
```

The whole protocol is **client ⇄ remote agent over HTTP/JSON-RPC**, organized around one durable unit of work — the **Task** — that moves through a lifecycle of states. Everything else (messages, parts, artifacts, streaming) hangs off that.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Agent Card** | A JSON discovery document, conventionally served at `/.well-known/agent-card.json`. Declares the agent's `name`, `description`, `url` (endpoint), `version`, `capabilities` (streaming, pushNotifications), `skills`, and `securitySchemes` (auth). This is how a client *discovers* and *vets* a remote agent before sending anything. |
| **Task** | The central, **stateful** unit of work, identified by a `taskId`. It moves through a lifecycle: `submitted` → `working` → (`input-required`) → `completed` / `failed` / `canceled`. Long-running tasks persist server-side so a client can poll or resubscribe. |
| **Message** | A single communication turn between client and agent (`role: "user"` or `"agent"`). A message carries one or more **Parts**. |
| **Part** | The atomic content unit. Three kinds: **TextPart** (plain text), **FilePart** (bytes or a URI), **DataPart** (structured JSON — e.g. a form). A2A is *modality-agnostic*: parts can be text, images, audio, forms. |
| **Artifact** | An **output** produced by the remote agent (also made of Parts). The tangible result of a task — a translated document, a chart, a JSON record. |
| **Streaming (SSE)** | If the agent advertises `streaming: true`, the client uses `message/stream` and receives **Server-Sent Events** with incremental status + artifact updates instead of one blocking response. |
| **Push notifications** | For very long tasks, the client registers a **webhook**; the agent POSTs updates there so the client needn't hold a connection open. |

**Transport & methods.** A2A rides on **HTTP(S)** with **JSON-RPC 2.0** as the default envelope (newer spec revisions also define gRPC and a plain HTTP+JSON/REST binding). The core RPC methods:

- `message/send` — send a message, get back a Task (or a direct Message) — blocking.
- `message/stream` — same, but the response is an SSE stream of updates.
- `tasks/get` — fetch the current state/artifacts of a task by id.
- `tasks/cancel` — request cancellation.
- `tasks/pushNotificationConfig/set` — register a webhook for a task.
- `tasks/resubscribe` — re-attach an SSE stream to an existing task after a dropped connection.

**Design principles** (worth remembering): build on **existing standards** (HTTP/SSE/JSON-RPC), keep agents **opaque** (no shared internals), be **secure by default** (enterprise auth via `securitySchemes`), and **support long-running, multi-modal** tasks natively.

## 4. Setup

A2A is language-agnostic — it's just JSON-RPC over HTTP, so any HTTP client/server can speak it. Google maintains an official **Python SDK**, `a2a-sdk`, plus a JS/TS SDK.

```bash
pip install a2a-sdk        # official Python SDK (server + client helpers)
# JS/TS:  npm install @a2a-js/sdk
```

The worked examples below are **pure-Python simulations of the wire protocol** — no installs, no network — so you can see exactly what flows between agents. The final cell shows the *real* SDK shape and runs only if `a2a-sdk` is installed.

In [1]:
# Installs are optional — the core examples are dependency-free and offline.
# Uncomment to get the real SDK used in the last cell:
# %pip install a2a-sdk

import importlib.util
print("a2a-sdk installed:", importlib.util.find_spec("a2a") is not None)


a2a-sdk installed: False


## 5. Worked Examples

### Example 1 — An Agent Card and a minimal A2A server, by hand

Discovery starts with the **Agent Card**. Then a client sends `message/send` and reads the resulting **Task**. Here's a toy "translator" agent: a JSON-RPC dispatcher that serves its card, accepts a task, and returns an artifact. Real servers add auth, streaming, and persistence — but the message shapes are exactly these.

In [2]:
import json

# --- The Agent Card: served at /.well-known/agent-card.json -----------------
AGENT_CARD = {
    "protocolVersion": "0.3.0",
    "name": "toy-translator",
    "description": "Translates short English text into French.",
    "url": "https://example.com/a2a",          # the JSON-RPC endpoint
    "version": "1.0.0",
    "capabilities": {"streaming": True, "pushNotifications": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "securitySchemes": {"bearer": {"type": "http", "scheme": "bearer"}},
    "skills": [
        {
            "id": "translate-en-fr",
            "name": "Translate EN->FR",
            "description": "Translate an English phrase to French.",
            "tags": ["translation", "language"],
            "examples": ["Translate 'good morning'"],
        }
    ],
}

# A tiny 'knowledge base' standing in for a real model.
_DICT = {"hello": "bonjour", "thank you": "merci", "good morning": "bonjour",
         "goodbye": "au revoir", "yes": "oui", "no": "non"}

TASKS = {}        # taskId -> task object (server-side state)
_next_id = [100]

def _text_of(message):
    """Pull the concatenated text from a message's parts."""
    return " ".join(p["text"] for p in message["parts"] if p["kind"] == "text")

def handle(request: dict) -> dict:
    """Dispatch one JSON-RPC 2.0 A2A request and return the response."""
    rid, method, params = request.get("id"), request["method"], request.get("params", {})

    if method == "agent/getCard":            # (illustrative; real cards are fetched over HTTP)
        result = AGENT_CARD

    elif method == "message/send":
        msg = params["message"]
        phrase = _text_of(msg).lower().strip(" .'\"")
        _next_id[0] += 1
        task_id = f"task-{_next_id[0]}"
        if phrase in _DICT:                  # we can complete immediately
            artifact = {
                "artifactId": f"art-{_next_id[0]}",
                "name": "translation",
                "parts": [{"kind": "text", "text": _DICT[phrase]}],
            }
            task = {"id": task_id, "contextId": "ctx-1",
                    "status": {"state": "completed"},
                    "artifacts": [artifact], "history": [msg]}
        else:                                # we need clarification from the client
            task = {"id": task_id, "contextId": "ctx-1",
                    "status": {"state": "input-required",
                               "message": {"role": "agent", "parts": [
                                   {"kind": "text",
                                    "text": f"I don't know '{phrase}'. Provide a known phrase."}]}},
                    "artifacts": [], "history": [msg]}
        TASKS[task_id] = task
        result = task

    elif method == "tasks/get":
        result = TASKS[params["id"]]

    else:
        return {"jsonrpc": "2.0", "id": rid,
                "error": {"code": -32601, "message": f"Method not found: {method}"}}

    return {"jsonrpc": "2.0", "id": rid, "result": result}


# --- Simulate a client session ----------------------------------------------
def user_message(text):
    return {"role": "user", "parts": [{"kind": "text", "text": text}]}

# 1. Discover the agent.
card = handle({"jsonrpc": "2.0", "id": 1, "method": "agent/getCard"})["result"]
print("Discovered agent:", card["name"], "- skills:", [s["id"] for s in card["skills"]])
print("Streaming supported:", card["capabilities"]["streaming"], "\n")

# 2. Send a task it can complete.
resp = handle({"jsonrpc": "2.0", "id": 2, "method": "message/send",
               "params": {"message": user_message("good morning")}})
task = resp["result"]
print("Task", task["id"], "->", task["status"]["state"])
print("Artifact:", task["artifacts"][0]["parts"][0]["text"])


Discovered agent: toy-translator - skills: ['translate-en-fr']
Streaming supported: True 

Task task-101 -> completed
Artifact: bonjour


### Example 2 — The Task lifecycle: an `input-required` round-trip

The Task is **stateful and durable**. When the agent can't proceed, it parks the task in `input-required` and waits. The client reads that state, supplies what's missing in a follow-up `message/send` (referencing the same `taskId`), and the agent resumes to `completed`. This multi-turn pattern is the heart of A2A — and what distinguishes it from a single tool call.

In [3]:
# 1. Send a phrase the agent does NOT know -> it asks for clarification.
resp = handle({"jsonrpc": "2.0", "id": 3, "method": "message/send",
               "params": {"message": user_message("see you later")}})
task = resp["result"]
task_id = task["id"]
print("Turn 1 -> state:", task["status"]["state"])
print("Agent asks:", task["status"]["message"]["parts"][0]["text"], "\n")

# 2. Client checks state independently via tasks/get (e.g. after a reconnect).
polled = handle({"jsonrpc": "2.0", "id": 4, "method": "tasks/get",
                 "params": {"id": task_id}})["result"]
print("tasks/get reports:", polled["status"]["state"], "\n")

# 3. Client supplies a phrase the agent knows -> task resumes to completed.
#    (Same conversation: a real client passes taskId/contextId to continue it.)
resp = handle({"jsonrpc": "2.0", "id": 5, "method": "message/send",
               "params": {"message": user_message("goodbye")}})
done = resp["result"]
print("Turn 2 -> state:", done["status"]["state"])
print("Final artifact:", done["artifacts"][0]["parts"][0]["text"])

# The lifecycle we just traversed:
print("\nTask states: submitted -> working -> input-required -> working -> completed")


Turn 1 -> state: input-required
Agent asks: I don't know 'see you later'. Provide a known phrase. 

tasks/get reports: input-required 

Turn 2 -> state: completed
Final artifact: au revoir

Task states: submitted -> working -> input-required -> working -> completed


### Example 3 — Streaming updates (SSE) and the real SDK

Long tasks shouldn't block. With `message/stream`, the server emits **Server-Sent Events** — incremental `TaskStatusUpdateEvent` / `TaskArtifactUpdateEvent` frames — until a `final` event. Below we *simulate* that event stream (no network), then show the canonical `a2a-sdk` server, which runs the introspection only if the SDK is installed.

In [4]:
# --- Simulate an SSE stream for a streaming task ----------------------------
def stream_translate(text):
    """Yield the status/artifact events a real A2A server would push over SSE."""
    phrase = text.lower().strip(" .'\"")
    yield {"kind": "status-update", "status": {"state": "submitted"}, "final": False}
    yield {"kind": "status-update", "status": {"state": "working"}, "final": False}
    result = _DICT.get(phrase, "??")
    yield {"kind": "artifact-update",
           "artifact": {"name": "translation", "parts": [{"kind": "text", "text": result}]},
           "final": False}
    yield {"kind": "status-update", "status": {"state": "completed"}, "final": True}

print("Client subscribes via message/stream and consumes SSE events:")
for event in stream_translate("hello"):
    if event["kind"] == "status-update":
        print(f"  event: status   -> {event['status']['state']:<14} final={event['final']}")
    else:
        print(f"  event: artifact -> {event['artifact']['parts'][0]['text']!r}")

# --- The real SDK shape ------------------------------------------------------
SNIPPET = """
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.agent_execution import AgentExecutor

card = AgentCard(
    name="toy-translator",
    description="Translates English to French.",
    url="https://example.com/a2a",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[AgentSkill(id="translate-en-fr", name="Translate EN->FR",
                       description="EN to FR.", tags=["translation"])],
)

class TranslatorExecutor(AgentExecutor):
    async def execute(self, context, event_queue):
        # read context.get_user_input(), enqueue artifact + status events
        ...
    async def cancel(self, context, event_queue):
        ...

app = A2AStarletteApplication(
    agent_card=card,
    http_handler=DefaultRequestHandler(agent_executor=TranslatorExecutor(),
                                       task_store=...),
).build()
# serve with: uvicorn module:app
"""

if importlib.util.find_spec("a2a") is not None:
    from a2a.types import AgentCard, AgentCapabilities, AgentSkill
    real = AgentCard(
        name="toy-translator",
        description="Translates English to French.",
        url="https://example.com/a2a",
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[AgentSkill(id="translate-en-fr", name="Translate EN->FR",
                           description="EN to FR.", tags=["translation"])],
    )
    print("\nBuilt a real AgentCard via a2a-sdk:", real.name, "v" + real.version)
else:
    print("\na2a-sdk not installed - here is the canonical server you would write:")
    print(SNIPPET)


Client subscribes via message/stream and consumes SSE events:
  event: status   -> submitted      final=False
  event: status   -> working        final=False
  event: artifact -> 'bonjour'
  event: status   -> completed      final=True

a2a-sdk not installed - here is the canonical server you would write:

from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.agent_execution import AgentExecutor

card = AgentCard(
    name="toy-translator",
    description="Translates English to French.",
    url="https://example.com/a2a",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[AgentSkill(id="translate-en-fr", name="Translate EN->FR",
                       description="EN to FR.", tags=["translation"])],
)

class TranslatorExecutor(AgentExe

## 6. Gotchas & Pitfalls

- **A2A is agent↔agent, not agent↔tool.** The single most common confusion. If you're exposing a *function/data source* to a model, that's **MCP**. A2A is for delegating to another *autonomous agent*. They stack: an A2A agent often uses MCP internally to reach its own tools.
- **The Task is stateful — design for it.** Unlike a stateless tool call, a Task lives server-side and can sit in `input-required` for a while. Persist tasks (a `TaskStore`), reference them by `taskId` across turns, and handle reconnection via `tasks/resubscribe`. Treating `message/send` as one-shot will break multi-turn flows.
- **Don't poll a streaming task in a tight loop.** If the agent advertises `streaming: true`, use `message/stream` (SSE) or register a **push-notification webhook** for long jobs — don't hammer `tasks/get`.
- **Agents are opaque on purpose — don't assume internals.** You can't reach into the remote agent's memory, tools, or prompt. All coordination happens through messages, parts, and artifacts. If you find yourself needing the other agent's internal state, A2A is the wrong boundary (or you should merge them).
- **Security is yours to wire up.** The Agent Card advertises `securitySchemes`, but you must actually authenticate (bearer/OAuth/API key) and authorize. Remote agents are untrusted code on the other end of a network — validate inputs and treat artifacts/messages as potentially adversarial (prompt-injection risk, just like with MCP servers).
- **Capabilities are advertised, not guaranteed-supported by the client.** Check the card before using `streaming` or `pushNotifications`; negotiate, don't assume.
- **The spec is young and moving.** A2A hit 1.0-era stability under the Linux Foundation in 2025, but method names and bindings (JSON-RPC vs gRPC vs REST) have evolved across revisions. Pin a `protocolVersion` and read the dated spec, don't trust memory.
- **Well-known path changed.** Newer revisions standardize on `/.well-known/agent-card.json`; older material references `/.well-known/agent.json`. Check which the peer serves.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-offs vs A2A |
|---|---|---|
| **A2A** | Composing **independent, possibly cross-vendor agents**; long-running, multi-modal, streaming tasks; preserving opacity between orgs | Adds HTTP + task-lifecycle machinery; pointless inside a single process/framework |
| **MCP** ([[model-context-protocol]]) | Giving an agent **tools, data, and prompts** | Different layer (agent↔tool, not agent↔agent). **Complementary** — agents speak A2A to each other and MCP to their tools |
| **In-framework multi-agent** ([[langgraph]], [[crewai]], [[autogen]]) | Orchestrating agents **inside one framework/process** with shared state | Tight coupling; no cross-vendor interop. Faster and simpler when you own all the agents. Several now *expose/consume* A2A at the edges |
| **Native function calling / handoffs** ([[openai-agents-sdk]]) | A single app delegating to sub-agents it owns | No standard wire format for *external* agents; you'd re-wrap per integration |
| **Plain REST / RPC between services** | Deterministic microservice calls | No notion of tasks, agent cards, streaming task updates, or multi-modal parts — you'd reinvent A2A's envelope |

**Rule of thumb:** if the thing you're calling is **another autonomous agent you don't fully control** (different team, vendor, or framework), reach for A2A. If it's a **tool or data source**, reach for MCP. If both agents live in **your** process, use your framework's native orchestration and skip the protocol overhead.

Related notebooks: [[model-context-protocol]], [[langgraph]], [[crewai]], [[autogen]], [[openai-agents-sdk]], [[semantic-kernel]].

## 8. Resources

- **Official site & docs** — https://a2a-protocol.org/
- **Specification (normative)** — https://a2a-protocol.org/latest/specification/
- **Python SDK (`a2a-sdk`)** — https://github.com/a2aproject/a2a-python
- **A2A project (samples, spec source)** — https://github.com/a2aproject/A2A
- **Google announcement (Apr 2025)** — https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/
- **Linux Foundation stewardship** — https://www.linuxfoundation.org/press/linux-foundation-launches-the-agent2agent-protocol-project
- **A2A + MCP, how they complement** — https://a2a-protocol.org/latest/topics/a2a-and-mcp/